# North DFW Housing Affordability Crisis
## Notebook 02: Data Cleaning & Wrangling

**Author:** Alejandro Molina  
**GitHub:** https://github.com/chooseyourtacoday  
**Date:** 2026

---

### Notebook Goal
Clean all staged datasets, resolve temporal alignment differences, and merge everything into a single master dataframe ready for feature engineering.

### Alignment Challenge
Our datasets arrive at different time frequencies:

| Dataset | Frequency | Coverage |
|---------|-----------|----------|
| Zillow Home Values | Monthly | 2015-2026 |
| Mortgage Rates | Monthly | 2015-2026 |
| Building Permits | Monthly | 2015-2026 |
| BLS Wages | Annual | 2015-2024 |
| Median Income | 5-Year Rolling | 2011-2023 |
| Population | Annual | 2020-2025 |

**Strategy:** Build a monthly spine from Zillow, then join annual and 5-year data using forward-fill interpolation.

## Step 1: Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os

STAGING_DIR = 'staging'
OUTPUT_DIR  = 'data/clean'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Libraries loaded.')
print(f'Staging : {STAGING_DIR}')
print(f'Output  : {OUTPUT_DIR}')

Libraries loaded.
Staging : staging
Output  : data/clean


## Step 2: Load All Staged Datasets

In [2]:
zillow     = pd.read_csv(os.path.join(STAGING_DIR, 'zillow_dfw_2015_present.csv'), parse_dates=['date'])
mortgage   = pd.read_csv(os.path.join(STAGING_DIR, 'mortgage_rates_monthly.csv'), parse_dates=['date'])
income     = pd.read_csv(os.path.join(STAGING_DIR, 'income_by_city.csv'))
population = pd.read_csv(os.path.join(STAGING_DIR, 'population_by_city.csv'))
permits    = pd.read_csv(os.path.join(STAGING_DIR, 'permits_dfw_monthly.csv'), parse_dates=['date'])
wages      = pd.read_csv(os.path.join(STAGING_DIR, 'bls_wages_annual.csv'))

print('All staging files loaded.')
print(f'  zillow     : {zillow.shape}')
print(f'  mortgage   : {mortgage.shape}')
print(f'  income     : {income.shape}')
print(f'  population : {population.shape}')
print(f'  permits    : {permits.shape}')
print(f'  wages      : {wages.shape}')

All staging files loaded.
  zillow     : (1755, 12)
  mortgage   : (136, 2)
  income     : (46, 3)
  population : (6, 7)
  permits    : (133, 9)
  wages      : (20, 5)


## Step 3: Clean Zillow Data
Zillow is our monthly spine — every other dataset joins to this.

In [3]:
# Keep only the columns we need
zillow_clean = zillow[['RegionName', 'city', 'CountyName', 'date', 'home_value']].copy()
zillow_clean.columns = ['zip_code', 'city', 'county', 'date', 'home_value']

# Add year and month columns for joining
zillow_clean['year']  = zillow_clean['date'].dt.year
zillow_clean['month'] = zillow_clean['date'].dt.month

print(f'Zillow clean shape : {zillow_clean.shape}')
print(f'Null values        : {zillow_clean.isnull().sum().sum()}')
print(f'Date range         : {zillow_clean["date"].min().date()} to {zillow_clean["date"].max().date()}')
zillow_clean.head(3)

Zillow clean shape : (1755, 7)
Null values        : 0
Date range         : 2015-01-31 to 2026-03-31


,zip_code,city,county,date,home_value,year,month
0,75002,Allen,Collin County,2015-01-31,236697.8775,2015,1
1,75002,Allen,Collin County,2015-02-28,239813.4993,2015,2
2,75002,Allen,Collin County,2015-03-31,243059.8287,2015,3


## Step 4: Clean Mortgage Rates

In [4]:
mortgage_clean = mortgage.copy()
mortgage_clean['year']  = mortgage_clean['date'].dt.year
mortgage_clean['month'] = mortgage_clean['date'].dt.month

print(f'Mortgage clean shape : {mortgage_clean.shape}')
print(f'Rate range           : {mortgage_clean["mortgage_rate"].min():.2f}% to {mortgage_clean["mortgage_rate"].max():.2f}%')
mortgage_clean.head(3)

Mortgage clean shape : (136, 4)
Rate range           : 2.68% to 7.62%


,date,mortgage_rate,year,month
0,2015-01-31,3.67,2015,1
1,2015-02-28,3.71,2015,2
2,2015-03-31,3.77,2015,3


## Step 5: Clean Building Permits
Permits are metro-wide (Dallas-Plano-Irving). We use total new single-family units as a supply pressure indicator.

In [5]:
permits_clean = permits[['date', 'units_1unit', 'units_5plus']].copy()
permits_clean['total_units'] = permits_clean['units_1unit'] + permits_clean['units_5plus']
permits_clean['year']  = permits_clean['date'].dt.year
permits_clean['month'] = permits_clean['date'].dt.month

# Check for nulls
print(f'Permits clean shape : {permits_clean.shape}')
print(f'Null values         : {permits_clean.isnull().sum().sum()}')
print(f'Date range          : {permits_clean["date"].min().date()} to {permits_clean["date"].max().date()}')
permits_clean.head(3)

Permits clean shape : (133, 6)
Null values         : 0
Date range          : 2015-01-01 to 2026-01-01


,date,units_1unit,units_5plus,total_units,year,month
0,2026-01-01,1817,1091,2908,2026,1
1,2025-12-01,1621,1909,3530,2025,12
2,2025-11-01,1530,800,2330,2025,11


## Step 6: Clean BLS Wage Data
Annual data. We assign county wages to cities based on which county each city falls in.
- Collin County (48085): Allen, Frisco (part), McKinney, Prosper, Celina, Plano
- Denton County (48121): Frisco (part)

Since most of our cities are in Collin County we use Collin as the primary wage reference, with Denton as a secondary comparison.

In [6]:
# Split into two county series
collin_wages = wages[wages['area_fips'] == 48085][['year', 'avg_weekly_wage', 'avg_annual_pay']].copy()
collin_wages.columns = ['year', 'collin_avg_weekly_wage', 'collin_avg_annual_pay']

denton_wages = wages[wages['area_fips'] == 48121][['year', 'avg_weekly_wage', 'avg_annual_pay']].copy()
denton_wages.columns = ['year', 'denton_avg_weekly_wage', 'denton_avg_annual_pay']

wages_clean = pd.merge(collin_wages, denton_wages, on='year', how='outer')

print(f'Wages clean shape : {wages_clean.shape}')
print(wages_clean.to_string(index=False))

Wages clean shape : (10, 5)
 year  collin_avg_weekly_wage  collin_avg_annual_pay  denton_avg_weekly_wage  denton_avg_annual_pay
 2015                    1185                  61631                     907                  47176
 2016                    1207                  62788                     935                  48644
 2017                    1233                  64123                     964                  50121
 2018                    1286                  66886                     976                  50734
 2019                    1315                  68366                     982                  51064
 2020                    1415                  73602                    1060                  55099
 2021                    1495                  77726                    1125                  58500
 2022                    1560                  81114                    1166                  60613
 2023                    1602                  83282                    

## Step 7: Clean Median Income Data
ACS 5-year rolling periods (e.g. 2017-2021). We extract the end year of each period as the reference year.

In [7]:
income_clean = income.copy()

# Extract end year from period string (e.g. '2017-2021' -> 2021)
income_clean['year'] = income_clean['period'].str.split('-').str[-1].astype(int)

# Keep most recent period per city
income_latest = income_clean.sort_values('year').drop_duplicates(subset='city', keep='last')

print(f'Income periods available:')
print(income_clean.groupby('city')['year'].agg(['min', 'max']))
print(f'\nMost recent income per city:')
print(income_latest[['city', 'median_income', 'year']].to_string(index=False))

Income periods available:
           min   max
city                
Allen     2015  2021
Celina    2015  2023
Frisco    2015  2021
McKinney  2015  2021
Plano     2015  2021
Prosper   2015  2023

Most recent income per city:
    city  median_income  year
McKinney         106437  2021
  Frisco         134210  2021
   Plano          99729  2021
   Allen         118254  2021
 Prosper         187603  2023
  Celina         155875  2023


## Step 8: Clean Population Data
Reshape from wide to long format for easier joining.

In [8]:
# Melt population from wide to long
pop_long = population.melt(
    id_vars='city',
    value_vars=['pop_2020', 'pop_2021', 'pop_2022', 'pop_2023', 'pop_2024', 'pop_2025'],
    var_name='year_label',
    value_name='population'
)

pop_long['year'] = pop_long['year_label'].str.extract(r'(\d{4})').astype(int)
pop_long = pop_long[['city', 'year', 'population']].sort_values(['city', 'year']).reset_index(drop=True)

print(f'Population long shape : {pop_long.shape}')
print(pop_long.head(12).to_string(index=False))

Population long shape : (36, 3)
  city  year  population
 Allen  2020    104627.0
 Allen  2021    107421.0
 Allen  2022    109169.0
 Allen  2023    110983.0
 Allen  2024    112155.0
 Allen  2025    112748.0
Celina  2020     16739.0
Celina  2021     23712.0
Celina  2022     29971.0
Celina  2023     38988.0
Celina  2024     44929.0
Celina  2025     47908.0


## Step 9: Build Master DataFrame
Start with Zillow monthly spine and join all other datasets.

**Join strategy:**
- Mortgage rates → join on year + month (exact monthly match)
- Permits → join on year + month (exact monthly match)
- Wages → join on year (annual, forward-filled to monthly)
- Population → join on year (annual, forward-filled to monthly)
- Income → join on city (most recent period used as static reference)

In [9]:
# Start with Zillow spine
master = zillow_clean.copy()
print(f'Base (Zillow)      : {master.shape}')

# Join mortgage rates
master = pd.merge(
    master,
    mortgage_clean[['year', 'month', 'mortgage_rate']],
    on=['year', 'month'],
    how='left'
)
print(f'After mortgage     : {master.shape}')

# Join permits
master = pd.merge(
    master,
    permits_clean[['year', 'month', 'units_1unit', 'units_5plus', 'total_units']],
    on=['year', 'month'],
    how='left'
)
print(f'After permits      : {master.shape}')

# Join wages (annual -> monthly via year)
master = pd.merge(
    master,
    wages_clean,
    on='year',
    how='left'
)
print(f'After wages        : {master.shape}')

# Join population (annual -> monthly via year)
master = pd.merge(
    master,
    pop_long,
    on=['city', 'year'],
    how='left'
)
print(f'After population   : {master.shape}')

# Join income (most recent period, static per city)
master = pd.merge(
    master,
    income_latest[['city', 'median_income']],
    on='city',
    how='left'
)
print(f'After income       : {master.shape}')

Base (Zillow)      : (1755, 7)
After mortgage     : (1755, 8)
After permits      : (1755, 11)
After wages        : (1755, 15)
After population   : (1755, 16)
After income       : (1755, 17)


## Step 10: Handle Missing Values

In [10]:
print('=== NULL CHECK BEFORE CLEANING ===')
print(master.isnull().sum())

=== NULL CHECK BEFORE CLEANING ===
zip_code                    0
city                        0
county                      0
date                        0
home_value                  0
year                        0
month                       0
mortgage_rate               0
units_1unit                26
units_5plus                26
total_units                26
collin_avg_weekly_wage    195
collin_avg_annual_pay     195
denton_avg_weekly_wage    195
denton_avg_annual_pay     195
population                819
median_income               0
dtype: int64


In [11]:
# Wages: forward fill for 2025-2026 months (no data yet beyond 2024)
wage_cols = ['collin_avg_weekly_wage', 'collin_avg_annual_pay',
             'denton_avg_weekly_wage', 'denton_avg_annual_pay']

master = master.sort_values(['zip_code', 'date'])
master[wage_cols] = master.groupby('zip_code')[wage_cols].ffill()

# Population: forward fill for months beyond 2025
master['population'] = master.groupby('zip_code')['population'].ffill()

# Population: back fill for months before 2020 (use 2020 census as baseline)
master['population'] = master.groupby('zip_code')['population'].bfill()

# Fill permit nulls with forward fill then back fill
permit_cols = ['units_1unit', 'units_5plus', 'total_units']
master[permit_cols] = master[permit_cols].ffill().bfill()

print('=== NULL CHECK FINAL ===')
print(master.isnull().sum())

=== NULL CHECK FINAL ===
zip_code                  0
city                      0
county                    0
date                      0
home_value                0
year                      0
month                     0
mortgage_rate             0
units_1unit               0
units_5plus               0
total_units               0
collin_avg_weekly_wage    0
collin_avg_annual_pay     0
denton_avg_weekly_wage    0
denton_avg_annual_pay     0
population                0
median_income             0
dtype: int64


## Step 11: Validate Master DataFrame

In [12]:
print('=== MASTER DATAFRAME SUMMARY ===')
print(f'Shape         : {master.shape}')
print(f'Date range    : {master["date"].min().date()} to {master["date"].max().date()}')
print(f'Zip codes     : {master["zip_code"].nunique()}')
print(f'Cities        : {sorted(master["city"].unique().tolist())}')
print(f'\nColumns       : {master.columns.tolist()}')
print(f'\nData types:')
print(master.dtypes)

=== MASTER DATAFRAME SUMMARY ===
Shape         : (1755, 17)
Date range    : 2015-01-31 to 2026-03-31
Zip codes     : 13
Cities        : ['Allen', 'Celina', 'Frisco', 'McKinney', 'Plano', 'Prosper']

Columns       : ['zip_code', 'city', 'county', 'date', 'home_value', 'year', 'month', 'mortgage_rate', 'units_1unit', 'units_5plus', 'total_units', 'collin_avg_weekly_wage', 'collin_avg_annual_pay', 'denton_avg_weekly_wage', 'denton_avg_annual_pay', 'population', 'median_income']

Data types:
zip_code                           int64
city                              object
county                            object
date                      datetime64[ns]
home_value                       float64
year                               int32
month                              int32
mortgage_rate                    float64
units_1unit                      float64
units_5plus                      float64
total_units                      float64
collin_avg_weekly_wage           float64
collin_avg_annu

In [13]:
# Spot check — one zip code, most recent 6 months
print('=== SPOT CHECK: Frisco 75034, Last 6 Months ===')
spot = master[
    (master['zip_code'] == 75034)
].sort_values('date').tail(6)
print(spot[['date', 'city', 'home_value', 'mortgage_rate', 
            'total_units', 'collin_avg_annual_pay', 
            'population', 'median_income']].to_string(index=False))

=== SPOT CHECK: Frisco 75034, Last 6 Months ===
      date   city  home_value  mortgage_rate  total_units  collin_avg_annual_pay  population  median_income
2025-10-31 Frisco 678549.7774         6.2540       3830.0                88091.0    240561.0         134210
2025-11-30 Frisco 679906.6461         6.2375       2330.0                88091.0    240561.0         134210
2025-12-31 Frisco 680861.3940         6.1900       3530.0                88091.0    240561.0         134210
2026-01-31 Frisco 680298.4085         6.1025       2908.0                88091.0    240561.0         134210
2026-02-28 Frisco 678663.7922         6.0475       2908.0                88091.0    240561.0         134210
2026-03-31 Frisco 676152.2558         6.1775       2908.0                88091.0    240561.0         134210


In [14]:
# Summary stats for key columns
print('=== KEY COLUMN SUMMARY ===')
key_cols = ['home_value', 'mortgage_rate', 'total_units', 
            'collin_avg_annual_pay', 'population', 'median_income']
print(master[key_cols].describe().round(2))

=== KEY COLUMN SUMMARY ===
       home_value  mortgage_rate  total_units  collin_avg_annual_pay  \
count     1755.00        1755.00      1755.00                1755.00   
mean    460777.45           4.71      3926.21               74464.24   
std     137949.98           1.44       834.53                9680.87   
min     214970.86           2.68      2302.00               61631.00   
25%     356376.86           3.62      3351.00               64123.00   
50%     434611.68           4.14      3823.00               73602.00   
75%     541110.59           6.27      4472.00               83282.00   
max     846894.91           7.62      6381.00               88091.00   

       population  median_income  
count     1755.00        1755.00  
mean    184419.60      123162.62  
std      86351.42       25005.01  
min      16739.00       99729.00  
25%     109169.00      106437.00  
50%     200509.00      118254.00  
75%     240561.00      134210.00  
max     294292.00      187603.00  


## Step 12: Save Master DataFrame

In [15]:
output_path = os.path.join(OUTPUT_DIR, 'master_dfw_housing.csv')
master.to_csv(output_path, index=False)

print('=' * 60)
print('NOTEBOOK 02 COMPLETE — DATA CLEANING SUMMARY')
print('=' * 60)
print(f'\nMaster dataframe saved to: {output_path}')
print(f'Rows    : {master.shape[0]}')
print(f'Columns : {master.shape[1]}')
print(f'\nColumn list:')
for col in master.columns:
    print(f'  {col}')
print('\nNext: Notebook 03 — Feature Engineering')

NOTEBOOK 02 COMPLETE — DATA CLEANING SUMMARY

Master dataframe saved to: data/clean\master_dfw_housing.csv
Rows    : 1755
Columns : 17

Column list:
  zip_code
  city
  county
  date
  home_value
  year
  month
  mortgage_rate
  units_1unit
  units_5plus
  total_units
  collin_avg_weekly_wage
  collin_avg_annual_pay
  denton_avg_weekly_wage
  denton_avg_annual_pay
  population
  median_income

Next: Notebook 03 — Feature Engineering
